In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="3"
import torch
from sentence_transformers import SentenceTransformer

# Each query needs to be accompanied by an corresponding instruction describing the task.
task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}

query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
queries = [
    'are judo throws allowed in wrestling?', 
    'how to become a radiology technician in michigan?'
    ]

# No instruction needed for retrieval passages
passages = [
    "Since you're reading this, you are probably someone from a judo background or someone who is just wondering how judo techniques can be applied under wrestling rules. So without further ado, let's get to the question. Are Judo throws allowed in wrestling? Yes, judo throws are allowed in freestyle and folkstyle wrestling. You only need to be careful to follow the slam rules when executing judo throws. In wrestling, a slam is lifting and returning an opponent to the mat with unnecessary force.",
    "Below are the basic steps to becoming a radiologic technologist in Michigan:Earn a high school diploma. As with most careers in health care, a high school education is the first step to finding entry-level employment. Taking classes in math and science, such as anatomy, biology, chemistry, physiology, and physics, can help prepare students for their college studies and future careers.Earn an associate degree. Entry-level radiologic positions typically require at least an Associate of Applied Science. Before enrolling in one of these degree programs, students should make sure it has been properly accredited by the Joint Review Committee on Education in Radiologic Technology (JRCERT).Get licensed or certified in the state of Michigan."
]

# load model with tokenizer
model = SentenceTransformer('nvidia/NV-Embed-v2', trust_remote_code=True)
model.max_seq_length = 1024
model.tokenizer.padding_side="right"

def add_eos(input_examples):
  input_examples = [input_example + model.tokenizer.eos_token for input_example in input_examples]
  return input_examples

# get the embeddings
batch_size = 2
query_embeddings = model.encode(add_eos(queries), batch_size=batch_size, prompt=query_prefix, normalize_embeddings=True)
passage_embeddings = model.encode(add_eos(passages), batch_size=batch_size, normalize_embeddings=True)

scores = (query_embeddings @ passage_embeddings.T) * 100
print(scores.tolist())


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
# file paths
embedding_easy_path='/raid/deallab/SF_RAG_Data/ASQA/train/train_embeddin_easy.csv'
embedding_hard_path='/raid/deallab/SF_RAG_Data/ASQA/train/train_embedding_hard.csv'

#load data
train_easy = pd.read_csv(embedding_easy_path)
print(train_easy.columns)

# Train/Val split
train_df, val_df = train_test_split(train_easy.loc[:10,:], test_size=0.2, random_state=42)

#convert to dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

#convert to right fromat
train_ds = train_ds.map(
    lambda row: {'anchor':row['question'], 'positive': row['text']}, 
    batched=False, 
    remove_columns=train_ds.column_names
)
val_ds = val_ds.map(
    lambda row: {'anchor':row['question'], 'positive': row['text']}, 
    batched=False, 
    remove_columns=val_ds.column_names
)

Index(['question', 'text'], dtype='object')


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [3]:
# Loss
train_loss = losses.MultipleNegativesRankingLoss(model)

#training_arguments
training_args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="/raid/deallab/SF_RAG_Data/ASQA/models/embedding/NV_embed/easy",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    fp16=True,  # Set to False if GPU can't handle FP16
    bf16=False,  # Set to True if GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicates
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
)

# Trainer 설정
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=train_loss
)

# train
trainer.train()

NameError: name 'losses' is not defined

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
# file paths
embedding_easy_path='/raid/deallab/SF_RAG_Data/ASQA/train/train_embeddin_easy.csv'
embedding_hard_path='/raid/deallab/SF_RAG_Data/ASQA/train/train_embedding_hard.csv'

#load data
train_hard = pd.read_csv(embedding_hard_path)
print(train_hard.columns)

# Train/Val split
train_df, val_df = train_test_split(train_hard, test_size=0.2, random_state=42)

#convert to dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

#convert to right fromat
train_ds = train_ds.map(
    lambda row: {'anchor':row['question'], 'positive': row['text_pos'],'negative': row['text_neg']}, 
    batched=False, 
    remove_columns=train_ds.column_names
)
val_ds = val_ds.map(
    lambda row: {'anchor':row['question'], 'positive': row['text_pos'],'negative': row['text_neg']}, 
    batched=False, 
    remove_columns=val_ds.column_names
)

4096

In [ ]:
# Loss
train_loss = losses.MultipleNegativesRankingLoss(model)

#training_arguments
training_args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="/raid/deallab/SF_RAG_Data/ASQA/models/embedding/NV_embed/easy",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    fp16=True,  # Set to False if GPU can't handle FP16
    bf16=False,  # Set to True if GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicates
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
)

# Trainer 설정
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=train_loss
)

# train
trainer.train()